In [1]:
from ipywidgets import IntProgress, Button, VBox, Output, Layout
from IPython.display import display, HTML
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.cm as cm

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from feature_engine.imputation import RandomSampleImputer

from sklearn.preprocessing import RobustScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
#  Стиль Кнопок
model_button_style = HTML("""
<style>
.widget-button {
    font-family: 'Unbounded';
    font-size: 10px;
    font-weight: bold;
    color: white;
    background: linear-gradient(to bottom, #3CB371, #008B8B);
    border: none;
    border-radius: 8px;
    box-shadow: 0 4px 6px rgba(0,0,0,0.3);
    transition: all 0.2s ease-in-out;
}
.widget-button:hover {
    box-shadow: 0 6px 10px rgba(0,0,0,0.4);
    transform: scale(1.1) translateY(-3px);
}
.widget-button:active {
    box-shadow: inset 0 2px 4px rgba(0,0,0,0.5);
    transform: translateY(2px);
}
</style>
""")
model_button_style

In [3]:
def progress():
    prgBar = IntProgress(min = 0, max = 100)
    display(prgBar)

    while prgBar.value < prgBar.max:
        prgBar.value = prgBar.value + 1
        time.sleep(0.1)
    
    print('Процесс завершен \n')

In [4]:
data_original = pd.read_excel('Folds5x2_pp_TP.xlsx', engine='openpyxl')
display(data_original.head())
data_original.shape

,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


(9568, 5)

In [5]:
feature_names = ['AT', 'V', 'AP', 'RH']

# AT → IterativeImputer (с линейной регрессией)
iter_imputer = IterativeImputer(estimator=LinearRegression(), random_state=42)

# V → KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)

# AP, RH → RandomSampleImputer (feature_engine)
random_imputer = RandomSampleImputer(random_state=42)

# Определяем группы признаков
iter_cols = ['AT']
knn_cols = ['V']
random_cols = ['AP', 'RH']

# ColumnTransformer для разных стратегий
preprocessor = ColumnTransformer(
    transformers=[
        ("iter", iter_imputer, iter_cols),
        ("knn", knn_imputer, knn_cols),
        ("rand", random_imputer, random_cols)
    ],
    remainder="passthrough"
)

# Удаление выбросов 
def outlier_filter(X, y=None):
    """Фильтрация выбросов по AP и RH"""
    mask = (X['AP'] > 996.850) & (X['AP'] < 1029.490) & \
           (X['RH'] > 31.085) & (X['RH'] < 117.045)
    return X.loc[mask].reset_index(drop=True), (y.loc[mask].reset_index(drop=True) if y is not None else None)

# Масштабирование
scaler = RobustScaler()

In [6]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=600,
                                                   learning_rate=0.1,
                                                   max_depth=6,
                                                   min_samples_leaf=2,
                                                   min_samples_split=2,
                                                   random_state=0),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
    'XGBoost': XGBRegressor(),
    'ANN': 'ANN'
}

In [7]:
# Датафрейм с метриками для дальнейшего сравнения
results_df = pd.DataFrame(columns=[
    'Model',
    'Train R2',
    'Test R2',
    'Train MAE',
    'Test MAE',
    'Train MSE',
    'Test MSE',
    'Train RMSE',
    'Test RMSE',   
])

In [8]:
# Метрики
def evaluate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

In [9]:
# Функция результата выбранной модели

def model_results_metric(model):
    
    # Инициализация прогрессбара
    prgBar = IntProgress(min=0, max=10)  # этапы
    display(prgBar)
    
    global preprocessor
    global scaler
    global data_original

    data = data_original.copy()
    print('Original data')
    display(data_original.head(5), data.shape)

    # 1 - Определение признаков и таргета
    #data = data.dropna().copy()
    #display(data.head(5), data.shape)
    X = data.drop(columns=['PE'])
    y = data['PE']
    prgBar.value += 1

    # 2 - Фильтрация выбросов
    X, y = outlier_filter(X, y)
    prgBar.value += 1

    # 3 - Разделение на обучающую и тестовую выборки
    x_train, x_test, y_train, y_test = train_test_split(X,
                                                        y, 
                                                        test_size=0.3,
                                                        random_state=0)

    prgBar.value += 1

    # 4 Создание пайплайна 
    pipeline = Pipeline(steps=[
            ("imputation", preprocessor),
            ("scaling", scaler),
            ('model', model)
        ])
    prgBar.value += 1
    
    if model == 'ANN':  
        print('1 STEP - Outliers Filter\n2 STEP - Imputation\n3 STEP - RobusSaler')
        
        ann = Sequential([
            Input(shape=(x_train.shape[1],)),
            Dense(64, activation='relu'),
            Dense(32, activation='relu'),
            Dense(1)
        ])
        
        # Препроцессинг вручную
        x_train_nn = scaler.fit_transform(preprocessor.fit_transform(x_train))
        x_test_nn = scaler.transform(preprocessor.transform(x_test))
    
        # 5 Компиляция и обучение модели
        ann.compile(optimizer='adam', loss='mse', metrics=['r2_score'])
        ann.fit(x_train_nn, y_train, batch_size=64, epochs=100, validation_data=(x_test_nn, y_test), verbose=0)
        prgBar.value += 1
    
        # 6 Предсказания
        y_train_pred = ann.predict(x_train_nn).flatten()
        y_test_pred = ann.predict(x_test_nn).flatten()     
    else:
        display('1 STEP - Outliers Filter', pipeline)
        # 5 Применение пайплайна
        pipeline.fit(x_train, y_train)
        prgBar.value += 1
        
        # 6 Предсказания
        y_train_pred = pipeline.predict(x_train)
        y_test_pred = pipeline.predict(x_test)
        
    prgBar.value += 1

    # 7 Метрики
    mae_train, mse_train, rmse_train, r2_train = evaluate_metrics(y_train, y_train_pred)
    mae_test, mse_test, rmse_test, r2_test = evaluate_metrics(y_test, y_test_pred)
    print()
    print('Train')
    print(f"R²: {r2_train:.4f}")
    print(f"MAE: {mae_train:.2f}")
    print(f"MSE: {mse_train:.2f}")
    print(f"RMSE: {rmse_train:.2f}")
    print()
    print('Test')
    print(f"R²: {r2_test:.4f}")
    print(f"MAE: {mae_test:.2f}")
    print(f"MSE: {mse_test:.2f}")
    print(f"RMSE: {rmse_test:.2f}")
    prgBar.value += 1
    
    # 8 Визуализация
    plt.figure(figsize=(10,8))
    sns.regplot(x=y_test_pred, y=y_test, marker='o', color='DarkSeaGreen', scatter_kws={'alpha': 0.5}, line_kws={'color':'black'})
    plt.xlabel('Predicted PE', fontsize=14)
    plt.ylabel('Actual PE', fontsize=14)
    model_name = 'ANN' if model == 'ANN' else model.__class__.__name__
    plt.title(f"Test - {model_name}", fontsize=22)
    plt.show()
    prgBar.value += 1
    print()

    # 9 Значения в таблицу результатов
    results_df.loc[len(results_df)] = [
        model_name,
        r2_train,
        r2_test,
        mae_train,
        mae_test,
        mse_train,
        mse_test,
        rmse_train,
        rmse_test
    ] 
    prgBar.value += 1
  
    # 10 Завершение
    time.sleep(0.1)
    prgBar.value += 1

In [10]:
# Функция запуска модели
def run_model(name, model):
    with output:
        output.clear_output()
        display(HTML(f"<div style='font-size: 30px;'>🔄 Модель: <b>{name}</b></div>"))
        model_results_metric(model)

In [11]:
# Функция отображения результата
def show_result():
    with output:
        output.clear_output()
        results = results_df.set_index('Model')
        metrics = ['R2', 'MAE', 'MSE', 'RMSE']
        colors = ['Gray', 'MediumSeaGreen']  # Train / Test

        fig, axs = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle("Metrics to compare", fontsize=20)

        for i, metric in enumerate(metrics):
            row, col = divmod(i, 2)
            ax = axs[row, col]

            # Собираем данные в один DataFrame
            df_plot = pd.DataFrame({
                'Train': results[f'Train {metric}'],
                'Test': results[f'Test {metric}']
            })
            
            # Определяем порядок сортировки
            ascending_order = True if metric == 'R2' else False

            df_plot = df_plot.sort_values(by='Test', ascending=ascending_order)
            df_plot.plot(kind='barh', ax=ax, color=colors, fontsize=12)
            ax.set_title(metric)
            ax.set_xlabel(metric)
            ax.legend(loc='lower left')

        # Функция для добавления подписей на горизонтальных столбцах
        def add_values_on_bars_horizontal(ax):
            for p in ax.patches:
                width = p.get_width()
                ax.text(
                    width,
                    p.get_y() + p.get_height() / 2.,
                    f'{width:.2f}',
                    ha='right', va='center', fontsize=9
                )
        
        for ax in axs.flatten():
            add_values_on_bars_horizontal(ax)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

In [12]:
# Функция очистки
def clear():
    global results_df
    results_df = pd.DataFrame(columns=results_df.columns)
    
    with output:
        output.clear_output()

In [13]:
output = Output()
buttons = []

for name, model in models.items():
    btn = Button(description=name,
                 layout=Layout(width='30%',
                               height='60px'),
                 style={'font_size': '20px'})
    
    btn.on_click(lambda b, n=name, m=model: run_model(n, m))
    buttons.append(btn)

result_button = Button(description='Show Metrics to Compare',
                       layout=Layout(width='25%',
                                     height='60px'),
                       style={'font_size': '15px',
                             'text_color': 'black'})

result_button.on_click(lambda b: show_result())
buttons.append(result_button)

clear_button = Button(description='🧹')
clear_button.on_click(lambda b: clear())
buttons.append(clear_button)

# Отображение
vbox = VBox(buttons + [output],
            layout=Layout(
                align_items='center'
            ))

display(vbox)